# `nanoddpm-pro`

>**Goal:** Diffusion models from scratch — unified DDIM/EDM, ɛ/v-prediction

From-scratch implementation of **Denoising Diffusion Probabilistic Model (DDPM)** for CIFAR-10 with:
- **Mini-UNet** with sinusoidal time/class embeddings
- **Classifier-Free Guidance (CFG)** for conditional generation
- **DDIM sampling**: discrete timesteps, deterministic reverse loop
- **EDM sampling**: continuous noise, Euler/Heun ODE solvers, preconditioning
- **Training targets**: noise prediction (`epsilon`) or v-prediction (`v`)
- **PCA-FID**: lightweight quality tracking (no InceptionV3)
<br><br>

---
## 1. Classifier-Free Guidance (CFG)
Standard conditional diffusion requires a separate classifier to steer generation. CFG removes that dependency by training a single model on **paired conditional/unconditional data**.

During training, we randomly drop the class label ~10% of the time, forcing the network to learn:
- `ε_θ(x_t, t, y)` : noise prediction with conditioning
- `ε_θ(x_t, t, ∅)` : noise prediction without conditioning

At inference, we combine them linearly:
$$
\hat{\epsilon} = \epsilon_\theta(x_t, t, \emptyset) + w \cdot \left( \epsilon_\theta(x_t, t, y) - \epsilon_\theta(x_t, t, \emptyset) \right)
$$
Where `w` (`cfg_scale`) controls the guidance strength:
- `w = 1.0` → standard conditional sampling
- `w = 3.0–7.5` → sharper, class-aligned outputs
- `w > 7.5` → artifacts & oversaturation

**Why it works:** `(ε_cond - ε_uncond)` points toward the class data manifold. Scaling it amplifies the "pull" without external gradients.

---
## 2. DDIM: Deterministic Fast Sampling
DDPM sampling is slow because it adds random noise at every reverse step. DDIM reparameterizes the reverse process as a **deterministic ODE**, allowing us to skip steps safely.

Starting from the DDPM reverse mean:
$$
\mu_\theta(x_t, t) = \frac{1}{\sqrt{\alpha_t}} \left( x_t - \frac{\beta_t}{\sqrt{1-\bar{\alpha}_t}} \epsilon_\theta(x_t, t) \right)
$$

We can predict the original image $x_0$ directly:
$$
x_0 = \frac{x_t - \sqrt{1-\bar{\alpha}_t} \epsilon_\theta}{\sqrt{\bar{\alpha}_t}}
$$

DDIM replaces the stochastic update with:
$$
x_{t-1} = \sqrt{\bar{\alpha}_{t-1}} x_0 + \sqrt{1 - \bar{\alpha}_{t-1} - \sigma_t^2} \epsilon_\theta + \sigma_t z
$$
Setting $\sigma_t = 0$ removes the random noise `z`, making the path deterministic. Because the trajectory is fixed, we can evaluate the network at a subset of steps (e.g., `t = 500, 250, 100, 50, 0`) and interpolate cleanly.

**Result:** 10–50 steps instead of 500–1000. Same weights, same training loop, just a swapped sampler.
<br><br>

---
## 3. EDM: Optimized Schedule & ODE Sampling (`Euler`/`Heun`)
EDM (Elucidating Diffusion Models) replaces heuristic noise schedules with a **continuous, mathematically grounded framework**. Instead of discrete timesteps, diffusion is treated as an ordinary differential equation (ODE) in noise space.

- **Log-Normal Noise Sampling:** Draw noise levels from `σ ~ exp(𝒩(P_mean, P_std²))`. This concentrates training on mid-noise regimes where gradients are most informative.
- **I/O Preconditioning:** Wrap the network with analytically derived scalars so it always sees normalized inputs/outputs, stabilizing gradients across the entire trajectory:
  $$
  c_{\text{in}} = \frac{1}{\sqrt{\sigma^2+1}}, \quad c_{\text{noise}} = \frac{\log\sigma}{4}
  $$
  $$
  D(x, \sigma) = c_{\text{skip}}(\sigma)\,x + c_{\text{out}}(\sigma)\,F\!\left(x \cdot c_{\text{in}}, c_{\text{noise}}\right)
  $$
  Where $c_{\text{skip}} = \frac{1}{\sigma^2+1}$ and $c_{\text{out}} = \frac{\sigma}{\sqrt{\sigma^2+1}}$.
- **Deterministic ODE Sampler:** The reverse process becomes a first-order ODE: $\frac{dx}{d\sigma} = \frac{x - D(x,\sigma)}{\sigma}$. Discretizing gives two solver options:

  **Euler (1st-order, fast):**
  $$
  x_{\text{next}} = x + (\sigma_{\text{next}} - \sigma) \cdot \frac{x - D(x,\sigma)}{\sigma}
  $$

  **Heun (2nd-order, quality):** Predictor-corrector refinement:
  $$
  \begin{aligned}
  x_{\text{pred}} &= x + (\sigma_{\text{next}} - \sigma) \cdot \frac{x - D(x,\sigma)}{\sigma} \quad \text{(predict)} \\
  x_{\text{next}} &= x + \frac{\sigma_{\text{next}} - \sigma}{2} \cdot \left( \frac{x - D(x,\sigma)}{\sigma} + \frac{x_{\text{pred}} - D(x_{\text{pred}},\sigma_{\text{next}})}{\sigma_{\text{next}}} \right) \quad \text{(correct)}
  \end{aligned}
  $$

### Training Targets: Noise (`ε`) vs `v`-Prediction
The network can be trained to predict different quantities. Both share the same architecture and ODE solvers, but differ in loss formulation and sampling decoding:

| Target | Training Objective | Sampling Decode | Why Use It? |
|--------|-------------------|-----------------|-------------|
| **Noise (`ε`)** | Predicts added noise. Loss weighted by $1/c_{\text{out}}^2$. | $x_0 \approx D(x_\sigma, \sigma)$ | Standard DDPM/EDM baseline. Simple & effective. |
| **`v`-Prediction** | Predicts $v = c_{\text{out}}\epsilon - c_{\text{skip}}x_0$. Uniform loss weighting. | $x_0 = c_{\text{skip}}x_\sigma - c_{\text{out}}D(x_\sigma, \sigma)$ | Used in SD2+, Imagen. Keeps gradient magnitudes uniform across all `σ`, improving stability at high noise. |

**Key Insight:** Once decoded to $x_0$, the **Euler/Heun solvers apply identically** to both targets. This cleanly decouples training objectives from the sampling loop.

**Tradeoff:** Heun requires 2× network calls per step but typically achieves 10–20% lower PCA-FID at the same step count. Use `--solver heun` for quality-critical runs, `--solver euler` for rapid iteration.

**Result:** No more `alpha_bar` or discrete timestep indexing. Training converges faster, gradients stay stable at higher resolutions, and sampling naturally supports arbitrary step counts with interchangeable solvers and targets.
<br><br>

----

### `EDM` vs `DDIM`

| Aspect | DDIM | EDM Sampler |
|--------|------|-------------|
| **Parameterization** | Discrete `t` steps with `α_t`/`β_t` schedule | Continuous noise levels `σ` (sigma) |
| **Math Foundation** | Reparameterizes DDPM reverse mean into a deterministic loop | Solves the diffusion ODE `dx/dσ = (x - D(x,σ))/σ` via Euler/Heun |
| **Step Skipping** | Requires re-deriving reverse equations for arbitrary step counts | Naturally supports any number of steps; just change the `σ` grid |
| **Stability** | Can drift or saturate at high resolutions without careful clipping | Preconditioning (`c_in`, `c_out`, `c_skip`) keeps gradients stable across all `σ` |

<br><br>

---
## 4. PCA-FID: Lightweight Quality Tracking
Standard FID uses Inception-V3 to extract 2048-d features—heavy, opaque, and overkill for 32×32 images. PCA-FID replaces the black-box extractor with **unsupervised dimensionality reduction**:

1. Flatten images to `[B, 3072]`
2. Compute top-32 principal components across real + generated batches
3. Project both sets into the 32D subspace
4. Run the standard FID formula on the projected means/covariances:

$$
\text{PCA-FID} = \|\mu_r - \mu_g\|^2 + \text{Tr}\!\left(\Sigma_r + \Sigma_g - 2\sqrt{\Sigma_r \Sigma_g}\right)
$$

**Why it's better for educational builds:**
- No external weights or network downloads
- Runs in <0.1s on CPU
- Captures global structure & color distribution
- Fully transparent: principal components are inspectable
<br><br>

---

## Imports & Config

In [ ]:
import torch, torch.nn as nn, torch.optim as optim, torchvision, torchvision.transforms as T
import matplotlib.pyplot as plt, numpy as np, math, json, torch.nn.functional as F
from tqdm import trange, tqdm
import ipywidgets as widgets
import time
import os

import base64
from IPython.display import Image, display

%matplotlib inline
torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# === CONFIG (EDM-Aligned) ===
EPOCHS = 20          # Training epochs
BATCH_SIZE = 32      # Samples per batch
RESIZE = 32          # Image resolution (16 for speed, 32 for quality)
CFG_SCALE = 4.0      # Classifier-Free Guidance strength

# DDIM-specific
T_STEPS = 1000          # Diffusion timesteps for DDIM

# EDM-specific
P_MEAN = -1.2        # Mean of log-normal noise distribution
P_STD = 1.2          # Std dev of log-normal noise distribution
SIGMA_MIN = 0.002    # Minimum noise level (near clean)
SIGMA_MAX = 80.0     # Maximum noise level (pure noise)
SAMPLE_STEPS = 20    # Discretization steps for EDM Euler sampler

print(f"▶ Device: {device} | Batch size: {BATCH_SIZE} | CFG: {CFG_SCALE}")

## DDIM & EDM

## Dataset Loading & Preparation

In [ ]:
transform = T.Compose([T.Resize(RESIZE), T.ToTensor(), T.Normalize([0.5]*3, [0.5]*3)])
dataset = torchvision.datasets.CIFAR10(root="./data", train=True, download=True, transform=transform)
loader = torch.utils.data.DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2, pin_memory=True)
real_batch, real_labels = next(iter(torch.utils.data.DataLoader(dataset, batch_size=256, shuffle=False)))
real_batch = real_batch.to(device)
print(f"▶ CIFAR-10 loaded. Shape: {dataset[0][0].shape}")

## Mini-UNet Model

In [ ]:
_res_block = """
graph TD
    A[Input x] --> B[Conv2D 3x3]
    B --> C[GroupNorm]
    C --> D[SiLU]

    E[Time t] --> F[Sinusoidal Emb]
    F --> G[MLP: SiLU + Linear]
    G --> H((+))

    I[Class y] --> J[Embedding]
    J --> K[Dropout 0.1]
    K --> H

    H --> L[Add to Feature Map]
    D --> L

    L --> M[Conv2D 3x3]
    M --> N[GroupNorm]
    N --> O[SiLU]

    A --> P[Skip: Conv 1x1 or Identity]
    P --> Q((+))
    O --> Q
    Q --> R[Output]
"""

_mini_UNET = """
graph TD
    A[Input: 3xHxW] --> D1[ResBlock: 3→ch]
    D1 -->|Skip d1| S1[(d1)]

    D1 --> P1[AvgPool 2x2]
    P1 --> D2[ResBlock: ch→ch*2]
    D2 -->|Skip d2| S2[(d2)]

    D2 --> P2[AvgPool 2x2]
    P2 --> M[ResBlock: ch*2→ch*2]

    M --> U1[Int: 2x Nearest]
    S2 --> C1[Concat]
    U1 --> C1
    C1 --> U1B[ResBlock: ch*3→ch]

    U1B --> U2[Int: 2x Nearest]
    S1 --> C2[Concat]
    U2 --> C2
    C2 --> U2B[ResBlock: ch*2→3]

    U2B --> OUT[Conv2D 1x1]
    OUT --> Z[Output: 3xHxW]
"""

In [ ]:
def render_mermaid(graph_code):
    graphbytes = graph_code.encode("utf-8")
    base64_bytes = base64.b64encode(graphbytes)
    base64_string = base64_bytes.decode("utf-8")
    display(Image(url="https://mermaid.ink/img/" + base64_string))

def render_mermaid_HW(graph_code, width=None, height=None):
    # Fix: Encode and decode using 'utf-8' to handle Unicode characters
    graphbytes = graph_code.encode("utf-8")
    base64_bytes = base64.b64encode(graphbytes)
    base64_string = base64_bytes.decode("utf-8")
    display(Image(url="https://mermaid.ink/img/" + base64_string, width=width, height=height))


In [ ]:
render_mermaid_HW(_res_block, width=600, height=600)

In [ ]:
render_mermaid_HW(_mini_UNET, width=300, height=860)

In [ ]:
def sinusoidal_embedding(t, dim, max_period=10000):
    half = dim // 2
    freqs = torch.exp(-math.log(max_period) * torch.arange(half, dtype=torch.float32, device=t.device) / half)
    args = t[:, None].float() * freqs[None, :]
    return torch.cat([torch.cos(args), torch.sin(args)], dim=1)

class ResBlock(nn.Module):
    def __init__(self, in_ch, out_ch, time_dim, num_classes=10, dropout=0.1):
        super().__init__()

        # If out_ch is 3, set num_groups to 3. Otherwise, use 8.
        num_groups_gn = min(8, out_ch)

        self.time_dim = time_dim # Store time_dim
        self.conv1 = nn.Conv2d(in_ch, out_ch, 3, padding=1)
        self.norm1 = nn.GroupNorm(num_groups_gn, out_ch)
        self.conv2 = nn.Conv2d(out_ch, out_ch, 3, padding=1)
        self.norm2 = nn.GroupNorm(num_groups_gn, out_ch)
        self.time_mlp = nn.Sequential(nn.SiLU(), nn.Linear(time_dim, out_ch))
        self.class_emb = nn.Embedding(num_classes, time_dim)
        self.class_proj = nn.Linear(time_dim, out_ch)  # Projects class emb to match out_ch
        self.dropout = nn.Dropout(dropout)
        self.skip = nn.Conv2d(in_ch, out_ch, 1) if in_ch != out_ch else nn.Identity()

    def forward(self, x, t, labels=None):
        h = F.silu(self.norm1(self.conv1(x)))
        t_emb = self.time_mlp(sinusoidal_embedding(t, self.time_dim))

        # Handle CFG: zero embedding for unconditional, projected + dropout for conditional
        if labels is None:
            c_emb = torch.zeros_like(t_emb)
        else:
            c_emb = self.class_proj(self.dropout(self.class_emb(labels)))

        h = h + (t_emb + c_emb)[:, :, None, None]
        return F.silu(self.norm2(self.conv2(h))) + self.skip(x)

class MiniUNet(nn.Module):
    def __init__(self, time_dim=128, ch=32):
        super().__init__()
        self.down1 = ResBlock(3, ch, time_dim)
        self.down2 = ResBlock(ch, ch*2, time_dim)
        self.mid = ResBlock(ch*2, ch*2, time_dim)
        self.up1 = ResBlock(ch*4, ch, time_dim)
        self.up2 = ResBlock(ch*2, 3, time_dim)
        self.out = nn.Conv2d(3, 3, 1)

    def forward(self, x, t, labels=None):
        d1 = self.down1(x, t, labels)
        d2 = self.down2(F.avg_pool2d(d1, 2), t, labels)
        x = self.mid(F.avg_pool2d(d2, 2), t, labels)
        x = self.up1(torch.cat([F.interpolate(x, scale_factor=2), d2], 1), t, labels)
        x = self.up2(torch.cat([F.interpolate(x, scale_factor=2), d1], 1), t, labels)
        return self.out(x)



## PCA-FID

In [ ]:
def pca_fid(real, gen, n_components=32):
    r, g = real.view(real.shape[0], -1).cpu().double(), gen.view(gen.shape[0], -1).cpu().double()
    data = torch.cat([r, g], 0)
    mean = data.mean(0, keepdim=True)
    U, S, V = torch.pca_lowrank(data - mean, q=n_components)
    proj = (data - mean) @ V
    r_p, g_p = proj[:real.shape[0]], proj[real.shape[0]:]
    mu_r, mu_g = r_p.mean(0), g_p.mean(0)
    var_r, var_g = r_p.var(0)+1e-6, g_p.var(0)+1e-6
    return ((mu_r-mu_g)**2).sum().item() + (var_r+var_g-2*torch.sqrt(var_r*var_g)).sum().item()


# DDIM

## DDIM Sampler (noise or v-prediction) & Forward Diffusion

In [ ]:
# === DDIM NOISE SCHEDULE (Discrete beta) ===
beta = torch.linspace(1e-4, 0.02, T_STEPS, device=device)
alpha = 1.0 - beta
alpha_bar = torch.cumprod(alpha, dim=0)

def forward_diffusion(x0, t):
    """Add noise to x0 at timestep t"""
    sqrt_ab = torch.sqrt(alpha_bar[t])[:, None, None, None]
    sqrt_1m = torch.sqrt(1.0 - alpha_bar[t])[:, None, None, None]
    eps = torch.randn_like(x0)
    return sqrt_ab * x0 + sqrt_1m * eps, eps

In [ ]:
#==========================================================================================
@torch.no_grad()
def sample_ddim(n, labels, cfg_scale=1.0, ddim_steps=50, target='epsilon'):
    """DDIM sampler with epsilon/v-prediction support"""
    model.eval()
    x = torch.randn(n, 3, RESIZE, RESIZE, device=device)
    step_size = max(1, T_STEPS // ddim_steps)

    for t in reversed(range(0, T_STEPS, step_size)):
        t_t = torch.full((n,), t, dtype=torch.long, device=device)
        eps_u = model(x, t_t, None)
        eps_c = model(x, t_t, labels)
        eps_pred = eps_u + cfg_scale * (eps_c - eps_u)  # ε_pred or v_pred

        ab = alpha_bar[t]
        sqrt_ab = torch.sqrt(ab)
        sqrt_1m = torch.sqrt(1.0 - ab)

        # Decode to x0 based on training target
        if target == 'epsilon':
            pred_x0 = (x - sqrt_1m * eps_pred) / sqrt_ab
        else:  # v-prediction: x0 = √ᾱ·x_t - √(1-ᾱ)·v_pred
            pred_x0 = sqrt_ab * x - sqrt_1m * eps_pred

        pred_x0 = torch.clip(pred_x0, -1.0, 1.0)

        a_prev = alpha[t - step_size] if t >= step_size else alpha[0]
        sqrt_ap = torch.sqrt(a_prev)
        sqrt_1map = torch.sqrt(1.0 - a_prev)

        # Standard DDIM deterministic update
        x = sqrt_ap * pred_x0 + sqrt_1map * eps_pred
        x = torch.clip(x, -1.0, 1.0)

    return x

## Model Training

In [ ]:
def run_training_ddim(epochs, cfg_scale, ddim_steps, target='epsilon'):
    """Train DDIM with epsilon or v-prediction target"""
    metrics_log = []
    print(f"▶ Training DDIM | Target: {target} | CFG: {cfg_scale} | Steps: {ddim_steps}")

    for epoch in trange(1, epochs + 1, desc="Training"):
        model.train()
        epoch_loss, count = 0, 0

        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            t = torch.randint(0, T_STEPS, (imgs.shape[0],), device=device)
            xt, eps = forward_diffusion(imgs, t)

            # CFG masking
            train_labels = labels.clone()
            train_labels[torch.rand_like(labels.float()) < 0.1] = -1
            cond_mask = train_labels != -1
            uncond_mask = ~cond_mask

            # Forward pass
            pred = torch.zeros_like(eps if target == 'epsilon' else imgs)
            if cond_mask.any():
                pred[cond_mask] = model(xt[cond_mask], t[cond_mask], train_labels[cond_mask])
            if uncond_mask.any():
                pred[uncond_mask] = model(xt[uncond_mask], t[uncond_mask], None)

            # Loss based on target
            if target == 'epsilon':
                loss = F.mse_loss(pred, eps)
            else:  # v-prediction
                sqrt_ab = torch.sqrt(alpha_bar[t])[:, None, None, None]
                sqrt_1m = torch.sqrt(1.0 - alpha_bar[t])[:, None, None, None]
                v_target = sqrt_ab * eps - sqrt_1m * imgs
                loss = F.mse_loss(pred, v_target)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * imgs.shape[0]
            count += imgs.shape[0]

        # Evaluation: sample with SAME target used for training
        model.eval()
        with torch.no_grad():
            gen = sample_ddim(128, real_labels[:128].to(device), cfg_scale, ddim_steps, target=target)
            fid = pca_fid(real_batch[:128], gen)

        metrics_log.append({'epoch': epoch, 'loss': epoch_loss/count, 'pca_fid': fid})
        print(f"  Epoch {epoch:02d} | Loss: {metrics_log[-1]['loss']:.4f} | PCA-FID ({target}): {fid:.2f}")


    with open('nanoddpm_ddim_metrics.json', 'w') as f:
        json.dump(metrics_log, f, indent=2)
    print("Metrics saved to nanoddpm_ddim_metrics.json")

    return metrics_log

## Visualization

In [ ]:
def plot_single_run_ddim(metrics_log, cfg_scale, ddim_steps=50, target='epsilon'):
    """Plot results for a single training run, showing both decoders at inference"""
    fig, axs = plt.subplots(1, 2, figsize=(10, 3))

    # Left: Training Loss
    axs[0].plot([m['epoch'] for m in metrics_log], [m['loss'] for m in metrics_log], marker='o')
    axs[0].set_title(f'Training Loss ({target})')
    axs[0].grid(alpha=0.3)

    # Right: PCA-FID (only the target you trained with)
    axs[1].plot([m['epoch'] for m in metrics_log], [m['pca_fid'] for m in metrics_log], marker='s', color='orange')
    axs[1].set_title(f'PCA-FID ({target} decoder) ↓ better')
    axs[1].grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

    # Show final samples decoded with BOTH formulas
    print(f"\nFinal samples (trained with {target}, CFG={cfg_scale}, steps={ddim_steps})")
    model.eval()
    with torch.no_grad():
        for dec in ['epsilon', 'v']:
            all_samples = []
            for cls in range(10):
                lbl = torch.full((2,), cls, dtype=torch.long, device=device)
                all_samples.append(sample_ddim(2, lbl, cfg_scale, ddim_steps, target=dec))
            grid = torchvision.utils.make_grid(torch.cat(all_samples), nrow=10, normalize=True, value_range=(-1, 1))
            plt.figure(figsize=(12, 2.5))
            plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
            plt.axis('off')
            plt.title(f'Generated Samples (decoder={dec})')
            plt.tight_layout()
            plt.show()

In [ ]:
def plot_training_dynamics(log_epsilon, log_v, cfg_scale, ddim_steps=50):
    """Compare training dynamics of epsilon vs v-prediction targets"""
    fig, axs = plt.subplots(1, 2, figsize=(12, 4))

    # Left: Training Loss Comparison
    axs[0].plot([m['epoch'] for m in log_epsilon], [m['loss'] for m in log_epsilon],
                label='epsilon target', color='blue', linewidth=1.5, marker='o', markersize=3)
    axs[0].plot([m['epoch'] for m in log_v], [m['loss'] for m in log_v],
                label='v-prediction target', color='orange', linewidth=1.5, marker='^', markersize=3)
    axs[0].set_title('Training Loss: epsilon vs v')
    axs[0].set_xlabel('Epoch')
    axs[0].set_ylabel('Loss (MSE)')
    axs[0].legend()
    axs[0].grid(alpha=0.3)

    # Right: PCA-FID Comparison (same decoder as training target)
    axs[1].plot([m['epoch'] for m in log_epsilon], [m['pca_fid'] for m in log_epsilon],
                label='epsilon target', color='blue', linewidth=1.5, marker='o', markersize=3)
    axs[1].plot([m['epoch'] for m in log_v], [m['pca_fid'] for m in log_v],
                label='v-prediction target', color='orange', linewidth=1.5, marker='^', markersize=3)
    axs[1].set_title('PCA-FID: epsilon vs v (↓ better)')
    axs[1].set_xlabel('Epoch')
    axs[1].set_ylabel('PCA-FID')
    axs[1].legend()
    axs[1].grid(alpha=0.3)

    plt.suptitle(f'Training Dynamics Comparison (DDIM, CFG={cfg_scale}, steps={ddim_steps})', y=1.02)
    plt.tight_layout()
    plt.show()

    # Print final metrics summary
    print(f"\nFinal Metrics (Epoch {log_epsilon[-1]['epoch']}):")
    print(f"  epsilon: Loss={log_epsilon[-1]['loss']:.4f} | FID={log_epsilon[-1]['pca_fid']:.2f}")
    print(f"  v-pred:  Loss={log_v[-1]['loss']:.4f} | FID={log_v[-1]['pca_fid']:.2f}")

def plot_final_samples_comparison(cfg_scale, ddim_steps=50):
    """Generate and plot final samples from both trained models side-by-side"""
    print(f"\nFinal Samples Comparison (CFG={cfg_scale}, DDIM steps={ddim_steps})")
    model.eval()

    with torch.no_grad():
        # Generate with epsilon-trained model (decoded with epsilon)
        gen_eps = sample_ddim(20, torch.arange(0, 10).repeat_interleave(2).to(device),
                             cfg_scale, ddim_steps, target='epsilon')
        # Generate with v-trained model (decoded with v)
        gen_v = sample_ddim(20, torch.arange(0, 10).repeat_interleave(2).to(device),
                           cfg_scale, ddim_steps, target='v')

        # Plot epsilon samples
        grid_eps = torchvision.utils.make_grid(gen_eps, nrow=10, normalize=True, value_range=(-1, 1))
        plt.figure(figsize=(12, 2.5))
        plt.imshow(grid_eps.permute(1, 2, 0).cpu().numpy())
        plt.axis('off')
        plt.title('Samples from epsilon-trained model (epsilon decoder)')
        plt.tight_layout()
        plt.show()

        # Plot v samples
        grid_v = torchvision.utils.make_grid(gen_v, nrow=10, normalize=True, value_range=(-1, 1))
        plt.figure(figsize=(12, 2.5))
        plt.imshow(grid_v.permute(1, 2, 0).cpu().numpy())
        plt.axis('off')
        plt.title('Samples from v-prediction-trained model (v decoder)')
        plt.tight_layout()
        plt.show()

In [ ]:
@torch.no_grad()
def sample_trajectory_ddim(n=9, steps_to_plot=[800, 600, 400, 200, 50, 1], cls_idx=5, cfg_scale=4.0, ddim_steps=50, target='epsilon'):
    model.eval()
    labels = torch.full((n,), cls_idx, dtype=torch.long, device=device)
    x = torch.randn(n, 3, RESIZE, RESIZE, device=device)
    trajectory_map = {t: None for t in steps_to_plot}
    step_size = max(1, T_STEPS // ddim_steps)

    for t in reversed(range(0, T_STEPS, step_size)):
        t_t = torch.full((n,), t, device=device, dtype=torch.long)
        eps_u = model(x, t_t, None)
        eps_c = model(x, t_t, labels)
        pred = eps_u + cfg_scale * (eps_c - eps_u)

        ab = alpha_bar[t]
        sqrt_ab = torch.sqrt(ab)
        sqrt_1m = torch.sqrt(1.0 - ab)

        x0 = (x - sqrt_1m * pred) / sqrt_ab if target == 'epsilon' else sqrt_ab * x - sqrt_1m * pred
        x0 = torch.clip(x0, -1.0, 1.0)

        a_prev = alpha[t - step_size] if t >= step_size else alpha[0]
        eps_imp = (x - sqrt_ab * x0) / sqrt_1m
        x = torch.sqrt(a_prev) * x0 + torch.sqrt(1.0 - a_prev) * eps_imp

        if t in trajectory_map:
          trajectory_map[t] = x.clone().cpu()

    valid_steps = sorted([t for t in trajectory_map if trajectory_map[t] is not None], reverse=True)
    valid_imgs = [trajectory_map[t] for t in valid_steps]
    if not valid_imgs:
      return

    fig, axes = plt.subplots(1, len(valid_steps), figsize=(15, 2))
    if len(valid_steps) == 1:
      axes = [axes]
    for i, (t_val, img) in enumerate(zip(valid_steps, valid_imgs)):
        grid = torchvision.utils.make_grid(img, nrow=3, normalize=True, value_range=(-1, 1))
        axes[i].imshow(grid.permute(1, 2, 0).numpy())
        axes[i].set_title(f"t={t_val} | {target}")
        axes[i].axis('off')
    plt.tight_layout(); plt.show()

@torch.no_grad()
def generate_ddim(cfg, steps, cls_idx, target='epsilon'):
    """
    Generate images with DDIM (noise vs v-prediction)
    Args:
    - cfg (float): CFG scale
    - steps (int): Number of discrete DDIM steps
    - cls_idx (int): Class index for conditional generation
    - target (str): 'epsilon' or 'v'
    """
    try:
        model.eval()
        n = 9
        labels = torch.full((n,), int(cls_idx), dtype=torch.long, device=device)
        x = sample_ddim(n, labels, float(cfg), int(steps), target=target)

        grid = torchvision.utils.make_grid(x, nrow=3, normalize=True, value_range=(-1, 1))
        plt.figure(figsize=(4, 4))
        plt.imshow(grid.cpu().permute(1, 2, 0).numpy())
        plt.title(f'Class {int(cls_idx)} | CFG={float(cfg)} | Steps={int(steps)} | {target}')
        plt.axis('off')
        plt.show()
        plt.close()
    except Exception as e:
      print(f"Generation error: {e}")


## Run DDIM

## Noise ($\varepsilon$)

In [ ]:
# === STEP 1: Train epsilon target ===
# Reset model/optimizer for clean start
model = MiniUNet(time_dim=128, ch=32).to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-3)
print(f"▶ Params: {sum(p.numel() for p in model.parameters()):,}")

log_epsilon = run_training_ddim(
    epochs=20,
    cfg_scale=4.0,
    ddim_steps=50,
    target='epsilon'
)


In [ ]:
# Plot epsilon training: loss + FID + final samples (both decoders)
plot_single_run_ddim(log_epsilon, cfg_scale=4.0, ddim_steps=50, target='epsilon')


In [ ]:
print("=== DDIM Trajectories: epsilon target ===")
for cls in [0, 3, 5, 8]:  # Sample a few representative classes
    print(f"\n--- Class {cls} ---")
    sample_trajectory_ddim(
        n=9,
        steps_to_plot=[800, 600, 400, 200, 50, 1],
        cls_idx=cls,
        cfg_scale=4.0,
        ddim_steps=50,
        target='epsilon'
    )


## v-prediction (`v`)

In [ ]:
# === STEP 2: Train v-prediction target ===
# Reset again for fair comparison
model = MiniUNet(time_dim=128, ch=32).to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-3)

log_v = run_training_ddim(
    epochs=20,
    cfg_scale=4.0,
    ddim_steps=50,
    target='v'
)


In [ ]:
plot_single_run_ddim(log_v, cfg_scale=4.0, ddim_steps=50, target='v')

In [ ]:
print("\n=== DDIM Trajectories: v-prediction target ===")
for cls in [0, 3, 5, 8]:  # Same classes for fair comparison
    print(f"\n--- Class {cls} ---")
    sample_trajectory_ddim(
        n=9,
        steps_to_plot=[800, 600, 400, 200, 50, 1],
        cls_idx=cls,
        cfg_scale=4.0,
        ddim_steps=50,
        target='v'
    )

## Noise vs v-prediction

In [ ]:
# === STEP 3: Compare training dynamics ===
plot_training_dynamics(log_epsilon, log_v, cfg_scale=4.0, ddim_steps=50)


In [ ]:
# === STEP 4 (Optional): Compare final sample quality ===
plot_final_samples_comparison(cfg_scale=4.0, ddim_steps=50)

In [ ]:
print("=== Full DDIM Comparison: cls_idx=5 ===")
for target in ['epsilon', 'v']:
    print(f"\n--- Target: {target} ---")
    sample_trajectory_ddim(
        n=9,
        steps_to_plot=[800, 600, 400, 200, 50, 1],
        cls_idx=5,  # Fixed class for direct comparison
        cfg_scale=4.0,
        ddim_steps=50,
        target=target
    )

In [ ]:
# 4. Widget
widgets.interact(
    generate_ddim,
    cfg=widgets.FloatSlider(value=4.0, min=1.0, max=7.5, step=0.5, description='CFG'),
    steps=widgets.IntSlider(value=50, min=10, max=100, step=5, description='DDIM Steps'),
    cls_idx=widgets.IntSlider(value=0, min=0, max=9, step=1, description='Class'),
    target=widgets.ToggleButtons(options=['epsilon', 'v'], description='Decoder:')
)


# EDM

## EDM Wrapper & Sampler (noise or v-prediction)

In [ ]:
class EDMWrapper(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x, sigma, labels=None):
        c_in = 1 / torch.sqrt(sigma**2 + 1)
        c_noise = torch.log(sigma) / 4
        out = self.model(x * c_in[:,None,None,None], c_noise, labels)
        c_skip, c_out = 1/(sigma**2+1), sigma/torch.sqrt(sigma**2+1)
        return c_skip[:,None,None,None]*x + c_out[:,None,None,None]*out

    def sample_sigmas(self, n, device, P_mean=-1.2, P_std=1.2):
        return torch.exp(P_mean + P_std * torch.randn(n, device=device))

#==========================================================================================
@torch.no_grad()
def edm_sampler(model, n, labels, cfg=1.0, steps=20, solver='euler', target='epsilon'):
    """
    EDM sampler with Euler/Heun solvers and epsilon/v-prediction support
    Args:
        model: EDMWrapper or raw model
        n: number of samples
        labels: class labels tensor
        cfg: classifier-free guidance scale
        steps: number of discretization steps
        solver: 'euler' (1st-order) or 'heun' (2nd-order)
        target: 'epsilon' for standard EDM or 'v' for v-prediction
    """
    sigmas = torch.linspace(80.0, 0.002, steps, device=device)
    x = torch.randn(n, 3, RESIZE, RESIZE, device=device) * sigmas[0]

    for i, (s, s_next) in enumerate(zip(sigmas[:-1], sigmas[1:])):
        s_vec = torch.full((n,), s, device=device)
        D_u = model(x, s_vec, None)
        D_c = model(x, s_vec, labels)
        D = D_u + cfg * (D_c - D_u)

        # Decode based on target parameter
        if target == 'v':
            c_skip = 1 / (s**2 + 1)
            c_out = s / torch.sqrt(s**2 + 1)
            x0_pred = c_skip * x - c_out * D
        else:
            x0_pred = D  # epsilon target: D is already x0_pred

        if solver == 'euler':
            x = x + (s_next - s) * (x - x0_pred) / s
        elif solver == 'heun':
            x_pred = x + (s_next - s) * (x - x0_pred) / s
            if s_next > 1e-5:
                s_next_vec = torch.full((n,), s_next, device=device)
                D_u_p = model(x_pred, s_next_vec, None)
                D_c_p = model(x_pred, s_next_vec, labels)
                D_p = D_u_p + cfg * (D_c_p - D_u_p)

                if target == 'v':
                    c_skip_p = 1 / (s_next**2 + 1)
                    c_out_p = s_next / torch.sqrt(s_next**2 + 1)
                    x0_pred_p = c_skip_p * x_pred - c_out_p * D_p
                else:
                    x0_pred_p = D_p

                x = x + (s_next - s) * ((x - x0_pred) / s + (x_pred - x0_pred_p) / s_next) / 2
            else:
                x = x_pred

        x = torch.clip(x, -1.0, 1.0)
    return x

## Model Training

In [ ]:
def run_training(epochs, cfg_scale, sample_steps, target='epsilon'):
    metrics_log = []
    print(f"▶ Training with target='{target}' | CFG={cfg_scale} | Steps={sample_steps}")

    for epoch in trange(1, epochs + 1, desc="Training"):
        model.train()
        epoch_loss, count = 0, 0

        for imgs, labels in loader:
            imgs, labels = imgs.to(device), labels.to(device)
            sigma = edm_wrapper.sample_sigmas(imgs.shape[0], device)
            x_sigma = imgs + sigma[:, None, None, None] * torch.randn_like(imgs)

            # 10% unconditional masking for CFG
            train_labels = labels.clone()
            train_labels[torch.rand_like(labels.float()) < 0.1] = -1

            cond_mask = train_labels != -1
            uncond_mask = ~cond_mask
            pred = torch.zeros_like(x_sigma)

            if cond_mask.any():
                pred[cond_mask] = edm_wrapper(x_sigma[cond_mask], sigma[cond_mask], train_labels[cond_mask])
            if uncond_mask.any():
                pred[uncond_mask] = edm_wrapper(x_sigma[uncond_mask], sigma[uncond_mask], None)

            # Loss calculation based on training target
            if target == 'v':
                c_skip = 1 / (sigma**2 + 1)
                c_out = sigma / torch.sqrt(sigma**2 + 1)
                eps = (x_sigma - imgs) / sigma[:, None, None, None]
                v_target = c_out[:, None, None, None] * eps - c_skip[:, None, None, None] * imgs
                loss = F.mse_loss(pred, v_target)  # Uniform weighting (standard for v-pred)
            else:
                c_out = sigma / torch.sqrt(sigma**2 + 1)
                loss_weight = (1.0 / (c_out ** 2)).view(-1, 1, 1, 1)
                loss = (loss_weight * F.mse_loss(pred, imgs, reduction='none')).mean()

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            epoch_loss += loss.item() * imgs.shape[0]
            count += imgs.shape[0]

        # === EVALUATION: Log all 4 solver/target combos ===
        model.eval()
        with torch.no_grad():
            # Decode as epsilon vs v during sampling
            fast_eps = edm_sampler(edm_wrapper, 128, real_labels[:128].to(device),
                                  cfg=cfg_scale, steps=sample_steps, solver='euler', target='epsilon')
            hard_eps = edm_sampler(edm_wrapper, 128, real_labels[:128].to(device),
                                  cfg=cfg_scale, steps=sample_steps, solver='heun', target='epsilon')
            fast_v = edm_sampler(edm_wrapper, 128, real_labels[:128].to(device),
                                cfg=cfg_scale, steps=sample_steps, solver='euler', target='v')
            hard_v = edm_sampler(edm_wrapper, 128, real_labels[:128].to(device),
                                cfg=cfg_scale, steps=sample_steps, solver='heun', target='v')

            metrics_log.append({
                'epoch': epoch,
                'loss': epoch_loss / count,
                'pca_fid_fast_epsilon': pca_fid(real_batch[:128], fast_eps),
                'pca_fid_hard_epsilon': pca_fid(real_batch[:128], hard_eps),
                'pca_fid_fast_v': pca_fid(real_batch[:128], fast_v),
                'pca_fid_hard_v': pca_fid(real_batch[:128], hard_v),
            })
            print(f"  Epoch {epoch:02d} | Loss: {metrics_log[-1]['loss']:.4f} | "
                  f"PCA-FID --> ε/E: {metrics_log[-1]['pca_fid_fast_epsilon']:.1f} | "
                  f"ε/H: {metrics_log[-1]['pca_fid_hard_epsilon']:.1f} | "
                  f"v/E: {metrics_log[-1]['pca_fid_fast_v']:.1f} | "
                  f"v/H: {metrics_log[-1]['pca_fid_hard_v']:.1f}")

    with open('nanoddpm_edm_metrics.json', 'w') as f:
        json.dump(metrics_log, f, indent=2)
    print("Metrics saved to nanoddpm_edm_metrics.json")

    return metrics_log

## Visualization

In [ ]:
def plot_fid_comparison(log_eps, log_v, solver='euler'):
    suffix = 'fast' if solver == 'euler' else 'hard'

    fig, ax = plt.subplots(1, 1, figsize=(6, 4))

    ax.plot([m['epoch'] for m in log_eps], [m[f'pca_fid_{suffix}_epsilon'] for m in log_eps],
            label='epsilon target', color='blue', linewidth=1.5, marker='o')
    ax.plot([m['epoch'] for m in log_v], [m[f'pca_fid_{suffix}_v'] for m in log_v],
            label='v-prediction target', color='orange', linewidth=1.5, marker='^')

    ax.set_title(f'PCA-FID Comparison ({solver.upper()} solver)')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('PCA-FID (↓ better)')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()

def plot_loss_comparison(log_eps, log_v):
    fig, ax = plt.subplots(1, 1, figsize=(6, 4))

    ax.plot([m['epoch'] for m in log_eps], [m['loss'] for m in log_eps],
            label='epsilon target', color='blue', linewidth=1.5, marker='o')
    ax.plot([m['epoch'] for m in log_v], [m['loss'] for m in log_v],
            label='v-prediction target', color='orange', linewidth=1.5, marker='^')

    ax.set_title('Training Loss: epsilon vs v-prediction')
    ax.set_xlabel('Epoch')
    ax.set_ylabel('Loss (MSE)')
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout()
    plt.show()


In [ ]:
def plot_results(metrics_log, cfg_scale, target='epsilon'):
    """Plot training metrics + final EDM-generated samples for both Euler and Heun"""
    fast_key = f'pca_fid_fast_{target}'
    hard_key = f'pca_fid_hard_{target}'
    name = target.upper()

    # === ROW 1: Loss + FID Comparison (Both Solvers) ===
    fig, axs = plt.subplots(1, 2, figsize=(12, 6))

    # Left: Training Loss (single curve, shared)
    axs[0].plot([m['epoch'] for m in metrics_log], [m['loss'] for m in metrics_log], marker='o')
    axs[0].set_title('Training Loss')
    axs[0].grid(alpha=0.3)
    axs[0].set_xlabel('Epoch')

    # Right: PCA-FID Comparison (Euler vs Heun)
    axs[1].plot([m['epoch'] for m in metrics_log], [m[fast_key] for m in metrics_log],
                marker='s', label='Euler (fast)', color='blue', linewidth=1.5)
    axs[1].plot([m['epoch'] for m in metrics_log], [m[hard_key] for m in metrics_log],
                marker='^', label='Heun (quality)', color='orange', linewidth=1.5)
    axs[1].set_title(f'PCA-FID: Euler vs Heun ({name}) ↓ better')
    axs[1].legend()
    axs[1].grid(alpha=0.3)
    axs[1].set_xlabel('Epoch')

    plt.tight_layout()
    plt.show()

    # === ROW 2: Final Samples - Euler (2 per class) ===
    print(f"\nFinal samples - Euler (CFG={cfg_scale}, target={target})")
    model.eval()
    with torch.no_grad():
        all_samples_euler = []
        for cls in range(10):
            labels = torch.full((2,), cls, dtype=torch.long, device=device)
            gen = edm_sampler(edm_wrapper, n=2, labels=labels, cfg=cfg_scale,
                            steps=SAMPLE_STEPS, solver='euler', target=target)
            all_samples_euler.append(gen)
        all_samples_euler = torch.cat(all_samples_euler, dim=0)

        grid = torchvision.utils.make_grid(all_samples_euler, nrow=10, normalize=True, value_range=(-1, 1))
        plt.figure(figsize=(12, 2.5))
        plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
        plt.axis('off')
        plt.title(f'Generated Samples - Euler (2 per class, {target})')
        plt.tight_layout()
        plt.show()

    # === ROW 3: Final Samples - Heun (2 per class) ===
    print(f"\nFinal samples - Heun (CFG={cfg_scale}, target={target})")
    with torch.no_grad():
        all_samples_heun = []
        for cls in range(10):
            labels = torch.full((2,), cls, dtype=torch.long, device=device)
            gen = edm_sampler(edm_wrapper, n=2, labels=labels, cfg=cfg_scale,
                            steps=SAMPLE_STEPS, solver='heun', target=target)
            all_samples_heun.append(gen)
        all_samples_heun = torch.cat(all_samples_heun, dim=0)

        grid = torchvision.utils.make_grid(all_samples_heun, nrow=10, normalize=True, value_range=(-1, 1))
        plt.figure(figsize=(12, 2.5))
        plt.imshow(grid.permute(1, 2, 0).cpu().numpy())
        plt.axis('off')
        plt.title(f'Generated Samples - Heun (2 per class, {target})')
        plt.tight_layout()
        plt.show()


def plot_compare_targets(metrics_log, cfg_scale, solver='euler'):
    """Compare epsilon vs v-prediction for a fixed solver (euler or heun) - NO LOSS PLOT"""
    suffix = 'fast' if solver == 'euler' else 'hard'
    name = f'{solver.upper()} solver'

    fig, ax = plt.subplots(1, 1, figsize=(6, 3))
    ax.plot([m['epoch'] for m in metrics_log], [m[f'pca_fid_{suffix}_epsilon'] for m in metrics_log],
            marker='s', label='epsilon', color='blue')
    ax.plot([m['epoch'] for m in metrics_log], [m[f'pca_fid_{suffix}_v'] for m in metrics_log],
            marker='^', label='v-prediction', color='orange')
    ax.set_title(f'PCA-FID: epsilon vs v ({name})')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xlabel('Epoch')
    plt.tight_layout()
    plt.show()

@torch.no_grad()
def compare_samplers(cls_idx=5, cfg=4.0, steps=20, target='epsilon'):
    """Generate same seed with Euler vs Heun for a specific training target"""
    model.eval()
    torch.manual_seed(42)  # Same noise for fair comparison

    labels = torch.full((4,), cls_idx, dtype=torch.long, device=device)

    # Euler samples
    euler = edm_sampler(edm_wrapper, n=4, labels=labels, cfg=cfg, steps=steps, solver='euler', target=target)

    # Heun samples (reset seed for identical starting noise)
    torch.manual_seed(42)
    heun = edm_sampler(edm_wrapper, n=4, labels=labels, cfg=cfg, steps=steps, solver='heun', target=target)

    # Plot
    fig, axs = plt.subplots(2, 4, figsize=(12, 3))
    for i in range(4):
        # Ensure values are clipped and normalized for display
        axs[0,i].imshow(euler[i].permute(1,2,0).cpu().numpy() * 0.5 + 0.5)
        axs[0,i].axis('off')
        axs[1,i].imshow(heun[i].permute(1,2,0).cpu().numpy() * 0.5 + 0.5)
        axs[1,i].axis('off')

    axs[0,0].set_title(f'Euler ({target})')
    axs[1,0].set_title(f'Heun ({target})')
    plt.suptitle(f'Class {cls_idx} | CFG={cfg} | Steps={steps}')
    plt.tight_layout()
    plt.show()

    # Quick metric: sample variance
    print(f"[{target.upper()}] Euler std: {euler.std().item():.4f} | Heun std: {heun.std().item():.4f}")

def plot_compare_separate_logs(log_eps, log_v, cfg_scale, solver='euler'):
    suffix = 'fast' if solver == 'euler' else 'hard'
    name = f'{solver.upper()} Solver'

    fig, ax = plt.subplots(1, 1, figsize=(6, 3))
    ax.plot([m['epoch'] for m in log_eps], [m[f'pca_fid_{suffix}_epsilon'] for m in log_eps],
            marker='s', label='epsilon target', color='blue')
    ax.plot([m['epoch'] for m in log_v], [m[f'pca_fid_{suffix}_v'] for m in log_v],
            marker='^', label='v-prediction target', color='orange')
    ax.set_title(f'PCA-FID: epsilon vs v ({name})')
    ax.legend()
    ax.grid(alpha=0.3)
    ax.set_xlabel('Epoch')
    plt.tight_layout()
    plt.show()


In [ ]:
@torch.no_grad()
def sample_trajectory(n=9, sigmas_to_plot=[80, 40, 20, 10, 2, 0.002], cls_idx=5, cfg_scale=4.0, steps=50, solver='euler', target='epsilon'):
    """Visualize denoising from high noise (σ=80) to clean (σ≈0)"""
    model.eval()
    labels = torch.full((n,), cls_idx, device=device)
    sigmas = torch.linspace(sigmas_to_plot[0], sigmas_to_plot[-1], steps, device=device)
    x = torch.randn(n, 3, RESIZE, RESIZE, device=device) * sigmas[0]  # Initialize noise (scaled by sigma_max)
    trajectory_map = {t: None for t in sigmas_to_plot}                # Map to store exactly one image per target sigma

    # Same edm_sampler logic but record intermediates
    for i, (s, s_next) in enumerate(zip(sigmas[:-1], sigmas[1:])):
        s_vec = torch.full((n,), s, device=device)
        D_u = edm_wrapper(x, s_vec, None)
        D_c = edm_wrapper(x, s_vec, labels)
        D = D_u + cfg_scale * (D_c - D_u)

        if target == 'v':
            c_skip = 1 / (s**2 + 1)
            c_out = s / torch.sqrt(s**2 + 1)
            x0_pred = c_skip * x - c_out * D
        else:
            x0_pred = D

        if solver == 'euler':
            x = x + (s_next - s) * (x - x0_pred) / s
        elif solver == 'heun':
            x_pred = x + (s_next - s) * (x - x0_pred) / s
            if s_next > 1e-5:
                s_next_vec = torch.full((n,), s_next, device=device)
                D_u_p = edm_wrapper(x_pred, s_next_vec, None)
                D_c_p = edm_wrapper(x_pred, s_next_vec, labels)
                D_p = D_u_p + cfg_scale * (D_c_p - D_u_p)
                if target == 'v':
                    c_skip_p = 1 / (s_next**2 + 1)
                    c_out_p = s_next / torch.sqrt(s_next**2 + 1)
                    x0_pred_p = c_skip_p * x_pred - c_out_p * D_p
                else:
                    x0_pred_p = D_p
                x = x + (s_next - s) * ((x - x0_pred) / s + (x_pred - x0_pred_p) / s_next) / 2
            else:
                x = x_pred

        x = torch.clip(x, -1.0, 1.0)

        # Record image if close to a target sigma
        for target_s in sigmas_to_plot:
            if abs(s.item() - target_s) < 2.0:
                trajectory_map[target_s] = x.clone().cpu()

    # Filter out missed targets and sort descending (Noise -> Clean)
    valid_sigmas = sorted([k for k, v in trajectory_map.items() if v is not None], reverse=True)
    valid_images = [trajectory_map[k] for k in valid_sigmas]

    if not valid_sigmas:
        print("No images captured. Try increasing 'steps' or 'tolerance'.")
        return

    fig, axes = plt.subplots(1, len(valid_sigmas), figsize=(15, 2))
    if len(valid_sigmas) == 1:  # Handle single subplot case
      axes = [axes]
    for i, (sigma_val, img) in enumerate(zip(valid_sigmas, valid_images)):
        grid = torchvision.utils.make_grid(img, nrow=3, normalize=True, value_range=(-1, 1))
        axes[i].imshow(grid.permute(1, 2, 0).numpy())
        axes[i].set_title(f"σ={sigma_val:.1f} | {solver.upper()} | {target}")
        axes[i].axis('off')
    plt.tight_layout()
    plt.show()


#========================================================================================
@torch.no_grad()
def generate(cfg, steps, cls_idx, solver='euler', target='epsilon'):
    """
    Generate images with EDM (Euler vs Heun ODE solver; noise vs v-prediction)
    Args:
    - cfg (float): CFG scale
    - steps (int): Number of EDM Euler steps
    - cls_idx (int): Class index for conditional generation
    - solver (str): 'euler' or 'heun'
    - target (str): 'epsilon' or 'v'
    """
    try:
        model.eval()
        n = 9
        cfg, steps, cls_idx = float(cfg), int(steps), int(cls_idx)
        labels = torch.full((n,), cls_idx, dtype=torch.long, device=device)
        x = edm_sampler(edm_wrapper, n=n, labels=labels, cfg=cfg, steps=steps, solver=solver, target=target)

        grid = torchvision.utils.make_grid(x, nrow=3, normalize=True, value_range=(-1, 1))
        plt.figure(figsize=(4, 4))
        plt.imshow(grid.cpu().permute(1, 2, 0).numpy())
        plt.title(f'Class {cls_idx} | CFG={cfg} | Steps={steps} | {solver.upper()} | {target}')
        plt.axis('off')
        plt.show()
        plt.close()
    except Exception as e:
        print(f"Generation error: {e}")

def generate_with_checkpoint(cfg, steps, cls_idx, solver='euler', target='epsilon'):
    """
    load model checkpoints &
    apply generate() to generate images with EDM (Euler vs Heun ODE solver; noise vs v-prediction)
    Args:
    - cfg (float): CFG scale
    - steps (int): Number of EDM Euler steps
    - cls_idx (int): Class index for conditional generation
    - solver (str): 'euler' or 'heun'
    - target (str): 'epsilon' or 'v'
    """
    load_checkpoint(target)  # Ensure correct weights are active
    generate(cfg, steps, cls_idx, solver=solver, target=target)

## Checkpoint Helpers

In [ ]:
def save_checkpoint(target_name):
    path = f"model_{target_name}.pth"
    torch.save(model.state_dict(), path)
    print(f"Saved {target_name} weights to {path}")

def load_checkpoint(target_name):
    path = f"model_{target_name}.pth"
    if not os.path.exists(path):
        raise FileNotFoundError(f"Checkpoint {path} not found. Train the model first.")
    model.load_state_dict(torch.load(path, map_location=device))
    print(f"Loaded {target_name} weights from {path}")

## Run EDM

## Noise ($\varepsilon$)

In [ ]:
print("=== PHASE 1: Training with epsilon target ===")
model = MiniUNet().to(device)
edm_wrapper = EDMWrapper(model).to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-3)
print(f"▶ Params: {sum(p.numel() for p in model.parameters()):,}")

metrics_log_epsilon = run_training(epochs=EPOCHS, cfg_scale=CFG_SCALE, sample_steps=SAMPLE_STEPS, target='epsilon')
save_checkpoint('epsilon')


In [ ]:
# Visualize epsilon results
plot_results(metrics_log_epsilon, cfg_scale=CFG_SCALE)


In [ ]:
# Compare epsilon samplers
print("\n=== Comparing Euler vs Heun samplers (epsilon target) ===")
for i in range(10):
    compare_samplers(cls_idx=i, cfg=CFG_SCALE, steps=50, target='epsilon')
    print("\n")

In [ ]:
print("=== Trajectories: epsilon target ===")
for solver in ['euler', 'heun']:
    print(f"\n--- {solver.upper()} solver ---")
    sample_trajectory(
        n=9,
        sigmas_to_plot=[80, 40, 20, 10, 2, 0.002],
        cls_idx=5,
        cfg_scale=4.0,
        steps=50,
        solver=solver,
        target='epsilon'
    )

## v-prediction (`v`)

In [ ]:
# Reset optimizer/model if you want a fresh start, otherwise this continues from previous state
model = MiniUNet().to(device)
edm_wrapper = EDMWrapper(model).to(device)
optimizer = optim.Adam(model.parameters(), lr=2e-3)
print(f"▶ Params: {sum(p.numel() for p in model.parameters()):,}")

print("=== PHASE 2: Training with v-prediction target ===")
metrics_log_v = run_training(epochs=EPOCHS, cfg_scale=CFG_SCALE, sample_steps=SAMPLE_STEPS, target='v')
save_checkpoint('v')


In [ ]:
# Visualize v results
plot_results(metrics_log_v, cfg_scale=CFG_SCALE)


In [ ]:
# Compare v-prediction samplers
print("\n=== Comparing Euler vs Heun samplers (v-prediction target) ===")
for i in range(10):
    compare_samplers(cls_idx=i, cfg=CFG_SCALE, steps=50, target='v')
    print("\n")

In [ ]:
print("\n=== Trajectories: v-prediction target ===")
for solver in ['euler', 'heun']:
    print(f"\n--- {solver.upper()} solver ---")
    sample_trajectory(
        n=9,
        sigmas_to_plot=[80, 40, 20, 10, 2, 0.002],
        cls_idx=5,
        cfg_scale=4.0,
        steps=50,
        solver=solver,
        target='v'
    )

## Noise vs v-prediction

In [ ]:
plot_loss_comparison(metrics_log_epsilon, metrics_log_v)

In [ ]:
# Compare with Euler solver
plot_fid_comparison(metrics_log_epsilon, metrics_log_v, solver='euler')
# Compare with Heun solver
plot_fid_comparison(metrics_log_epsilon, metrics_log_v, solver='heun')

In [ ]:
# print("=== PHASE 3: Comparing epsilon vs v-prediction targets for Heun and Euler ODE solvers ===")
# load_checkpoint('epsilon')
# load_checkpoint('v')
# plot_compare_separate_logs(metrics_log_epsilon, metrics_log_v, CFG_SCALE, solver='euler')
# plot_compare_separate_logs(metrics_log_epsilon, metrics_log_v, CFG_SCALE, solver='heun')

In [ ]:
print("=== Full 2x2 Comparison: cls_idx=3 ===")
for target in ['epsilon', 'v']:
    for solver in ['euler', 'heun']:
        print(f"\n--- {target} target + {solver} solver ---")
        sample_trajectory(
            n=9,
            sigmas_to_plot=[80, 40, 20, 10, 2, 0.002],
            cls_idx=3,  # Fixed class for fair comparison
            cfg_scale=4.0,
            steps=50,
            solver=solver,
            target=target
        )

In [ ]:
widgets.interact(
    generate_with_checkpoint,
    cfg=widgets.FloatSlider(value=4.0, min=1.0, max=7.5, step=0.5, description='CFG Scale'),
    steps=widgets.IntSlider(value=20, min=10, max=50, step=5, description='EDM Steps'),
    cls_idx=widgets.IntSlider(value=0, min=0, max=9, step=1, description='Class (0-9)'),
    solver=widgets.ToggleButtons(options=['euler', 'heun'], description='Solver:'),
    target=widgets.ToggleButtons(options=['epsilon', 'v'], description='Target:')
)